# Phase 5 Evaluation Orchestration Walkthrough

This notebook walks through the run-level evaluation orchestration, failure classification, and evaluation analytics implemented in Phase 5.

It uses the implementation directly:

- `POST /v1/runs/{run_id}/evaluate` runs the full deterministic evaluator registry synchronously.
- `EvaluatorExecutionStatus` records evaluator infrastructure status independently from pass/fail labels.
- `FailureClassifier` converts evaluator findings into the current run-level `RunFailure` snapshot.
- `evaluation_results` keeps append-only evaluator invocation history.
- `run_failures` keeps the latest per-run evaluation/failure snapshot.
- `/v1/runs`, `/v1/runs/{run_id}`, `/v1/analytics/failures`, and `/v1/analytics/overview` expose the new read side.

Prerequisites from the repository root:

```bash
cp .env.example .env
docker compose up -d --wait postgres
uv run alembic upgrade head
```


## 1. Imports And Notebook Helpers

The helpers below are notebook-only setup, cleanup, and display code. Evaluation, ingestion, persistence, routing, and analytics are all imported from the application.


In [ ]:
from collections.abc import AsyncIterator, Iterable
from contextlib import asynccontextmanager
from dataclasses import asdict, is_dataclass
from datetime import UTC, datetime
from decimal import Decimal
import json
from pathlib import Path
import sys
from typing import Any

from IPython.display import Markdown, display
from httpx import ASGITransport, AsyncClient
from sqlalchemy import delete, func, select
from sqlalchemy.ext.asyncio import AsyncSession, async_sessionmaker

repo_root = Path.cwd()
if not (repo_root / "pyproject.toml").exists():
    repo_root = repo_root.parent

src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from obs_platform.config import DatabaseOnlySettings
from obs_platform.database import create_engine, wait_for_database
from obs_platform.db.models import (
    AgentRun,
    EvaluationResult as EvaluationResultRecord,
    LLMCall,
    RunFailure,
    Span,
    ToolCall,
)
from obs_platform.evaluation.classifier import (
    FAILURE_TYPE_PRIORITY,
    FAILURE_TYPE_SEVERITY,
    EvaluatorOutcome,
    FailureClassifier,
)
from obs_platform.evaluation.registry import DETERMINISTIC_EVALUATORS
from obs_platform.evaluation.types import (
    EvaluationFinding,
    EvaluationResult,
    EvaluatorExecutionStatus,
)
from obs_platform.ingestion.runs import ingest_run_event
from obs_platform.main import create_app
from obs_platform.routes import analytics, runs
from obs_platform.telemetry.v1 import load_fixture


DEMO_PREFIX = "notebook-phase-5-"
NEVER_EVALUATED_RUN_ID = f"{DEMO_PREFIX}never-evaluated"
DEMO_RUN_IDS = [
    f"{DEMO_PREFIX}healthy",
    f"{DEMO_PREFIX}tool-failure",
    f"{DEMO_PREFIX}trajectory-error",
    f"{DEMO_PREFIX}policy-violation",
    f"{DEMO_PREFIX}retrieval-failure",
    NEVER_EVALUATED_RUN_ID,
]
PRIMARY_DEMO_RUN_ID = f"{DEMO_PREFIX}policy-violation"


def jsonable(value: Any) -> Any:
    if is_dataclass(value):
        return asdict(value)
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if isinstance(value, Decimal):
        return float(value)
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, list):
        return [jsonable(item) for item in value]
    if isinstance(value, dict):
        return {key: jsonable(item) for key, item in value.items()}
    return value


def table(rows: Iterable[dict[str, Any]]) -> None:
    rows = list(rows)
    if not rows:
        display(Markdown("_No rows._"))
        return
    headers = list(rows[0])
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in rows:
        values = [json.dumps(jsonable(row.get(header)), sort_keys=True) for header in headers]
        lines.append("| " + " | ".join(values) + " |")
    display(Markdown("\n".join(lines)))


def finding(code: str) -> EvaluationFinding:
    return EvaluationFinding(
        code=code,
        message=f"Synthetic notebook finding: {code}",
        data={"source": "phase_5_walkthrough"},
    )


def completed_outcome(
    evaluator_name: str,
    *,
    label: str,
    findings: list[EvaluationFinding] | None = None,
    severity: str | None = None,
) -> EvaluatorOutcome:
    return EvaluatorOutcome(
        evaluator_name=evaluator_name,
        evaluator_version="notebook",
        execution_status=EvaluatorExecutionStatus.COMPLETED,
        result=EvaluationResult(
            passed=(label != "fail"),
            score=1.0 if label == "pass" else 0.0 if label == "fail" else None,
            label=label,
            severity=severity,
            reason=f"Synthetic {label} outcome for notebook demonstration.",
            findings=findings or [],
        ),
    )


def failed_outcome(evaluator_name: str) -> EvaluatorOutcome:
    return EvaluatorOutcome(
        evaluator_name=evaluator_name,
        evaluator_version="notebook",
        execution_status=EvaluatorExecutionStatus.FAILED,
        result=EvaluationResult(
            passed=False,
            score=None,
            label=None,
            severity=None,
            reason="RuntimeError: synthetic evaluator exception",
            findings=[finding("evaluator_exception")],
        ),
    )


def classification_row(name: str, outcomes: list[EvaluatorOutcome]) -> dict[str, Any]:
    classification = FailureClassifier().classify(outcomes)
    return {
        "case": name,
        "overall_status": classification.overall_status.value,
        "primary_category": (
            classification.primary_category.value if classification.primary_category else None
        ),
        "secondary_category": (
            classification.secondary_category.value if classification.secondary_category else None
        ),
        "max_severity": classification.max_severity,
    }


def compact_evaluator_result(row: dict[str, Any]) -> dict[str, Any]:
    return {
        "evaluator_name": row["evaluator_name"],
        "execution_status": row["execution_status"],
        "label": row["label"],
        "severity": row["severity"],
        "finding_codes": [item["code"] for item in row["findings"]],
    }


settings = DatabaseOnlySettings()
engine = create_engine(settings.db)
Session = async_sessionmaker(engine, expire_on_commit=False)


async def delete_demo_runs(session: AsyncSession, run_ids: list[str] = DEMO_RUN_IDS) -> None:
    await session.execute(delete(EvaluationResultRecord).where(EvaluationResultRecord.run_id.in_(run_ids)))
    await session.execute(delete(RunFailure).where(RunFailure.run_id.in_(run_ids)))
    await session.execute(delete(ToolCall).where(ToolCall.run_id.in_(run_ids)))
    await session.execute(delete(LLMCall).where(LLMCall.run_id.in_(run_ids)))
    await session.execute(delete(Span).where(Span.run_id.in_(run_ids)))
    await session.execute(delete(AgentRun).where(AgentRun.run_id.in_(run_ids)))
    await session.commit()


@asynccontextmanager
async def notebook_client() -> AsyncIterator[AsyncClient]:
    async def notebook_session() -> AsyncIterator[AsyncSession]:
        async with Session() as session:
            yield session

    app = create_app()
    app.dependency_overrides[runs.get_session] = notebook_session
    app.dependency_overrides[analytics.get_session] = notebook_session
    async with AsyncClient(
        transport=ASGITransport(app=app),
        base_url="http://testserver",
    ) as client:
        yield client


## 2. Evaluation Statuses And Failure Taxonomy

Phase 5 adds orchestration vocabulary around the Phase 4 evaluator results. Execution status answers whether the evaluator itself ran cleanly. Failure type and severity answer what kind of agent behavior failed after completed evaluator results are classified.


In [ ]:
table([
    {"execution_status": status.value}
    for status in EvaluatorExecutionStatus
])

table([
    {
        "priority": index,
        "failure_type": failure_type.value,
        "mapped_severity": FAILURE_TYPE_SEVERITY[failure_type],
    }
    for index, failure_type in enumerate(FAILURE_TYPE_PRIORITY, start=1)
])


## 3. Classifier Rules Without A Database

The classifier consumes the current in-memory outcomes from one evaluation orchestration call. It does not query historical `evaluation_results` rows.


In [ ]:
classifier_examples = [
    classification_row(
        "all completed pass/not_applicable",
        [
            completed_outcome("tool_execution", label="pass"),
            completed_outcome("trajectory", label="not_applicable"),
        ],
    ),
    classification_row(
        "completed evaluator reports fail",
        [
            completed_outcome("tool_execution", label="fail", findings=[finding("tool_call_failed")]),
            completed_outcome("structured_output", label="pass"),
        ],
    ),
    classification_row(
        "evaluator infrastructure failure only",
        [
            completed_outcome("tool_execution", label="pass"),
            failed_outcome("policy"),
        ],
    ),
    classification_row(
        "policy violation takes primary precedence",
        [
            completed_outcome("policy", label="fail", findings=[finding("unauthorized_consequential_action")]),
            completed_outcome("tool_execution", label="fail", findings=[finding("tool_call_error")]),
        ],
    ),
    classification_row(
        "two non-policy failures use static priority",
        [
            completed_outcome("trajectory", label="fail", findings=[finding("ordering_violation")]),
            completed_outcome("tool_execution", label="fail", findings=[finding("tool_call_failed")]),
        ],
    ),
]

table(classifier_examples)


## 4. Prepare Demo Runs

This section connects to PostgreSQL, removes prior notebook demo rows, ingests five canonical fixtures under notebook-specific run IDs, and evaluates each run through the real HTTP endpoint. It also ingests one extra run without evaluating it, so later analytics examples can show that never-evaluated runs are excluded.


In [ ]:
await wait_for_database(engine)

fixture_plan = [
    ("healthy_success", f"{DEMO_PREFIX}healthy"),
    ("tool_failure", f"{DEMO_PREFIX}tool-failure"),
    ("trajectory_error", f"{DEMO_PREFIX}trajectory-error"),
    ("policy_violation", f"{DEMO_PREFIX}policy-violation"),
    ("retrieval_failure", f"{DEMO_PREFIX}retrieval-failure"),
]

async with Session() as session:
    await delete_demo_runs(session)

async with notebook_client() as client:
    responses = []
    for fixture_name, run_id in fixture_plan:
        event = load_fixture(fixture_name).model_copy(update={"run_id": run_id})
        ingest_response = await client.post("/v1/runs", json=event.model_dump(mode="json"))
        evaluate_response = await client.post(f"/v1/runs/{run_id}/evaluate")
        responses.append({
            "fixture": fixture_name,
            "run_id": run_id,
            "ingest_status": ingest_response.status_code,
            "evaluate_status": evaluate_response.status_code,
            "overall_status": evaluate_response.json()["overall_status"],
            "failure": evaluate_response.json()["failure"],
        })

    unevaluated_event = load_fixture("healthy_success").model_copy(
        update={"run_id": NEVER_EVALUATED_RUN_ID}
    )
    unevaluated_response = await client.post(
        "/v1/runs",
        json=unevaluated_event.model_dump(mode="json"),
    )
    responses.append({
        "fixture": "healthy_success",
        "run_id": NEVER_EVALUATED_RUN_ID,
        "ingest_status": unevaluated_response.status_code,
        "evaluate_status": None,
        "overall_status": None,
        "failure": None,
    })

table(responses)


## 5. Inspect One Evaluation Response

`POST /v1/runs/{run_id}/evaluate` returns a dedicated trigger response. It contains the run-level verdict and one summary per evaluator invocation, but it does not include the full run detail payload.


In [ ]:
async with notebook_client() as client:
    response = await client.post(f"/v1/runs/{PRIMARY_DEMO_RUN_ID}/evaluate")
    body = response.json()

print(json.dumps({
    "run_id": body["run_id"],
    "overall_status": body["overall_status"],
    "failure": body["failure"],
    "evaluated_at": body["evaluated_at"],
}, indent=2))

table(compact_evaluator_result(item) for item in body["evaluator_results"])


## 6. Inspect Persistence: History Versus Snapshot

Every evaluator invocation appends to `evaluation_results`. The latest `run_failures` row is upserted once per run, so it represents the current classified snapshot.


In [ ]:
async with Session() as session:
    history_rows = (
        await session.execute(
            select(
                EvaluationResultRecord.id,
                EvaluationResultRecord.evaluator_name,
                EvaluationResultRecord.status,
                EvaluationResultRecord.label,
                EvaluationResultRecord.severity,
                EvaluationResultRecord.created_at,
            )
            .where(EvaluationResultRecord.run_id == PRIMARY_DEMO_RUN_ID)
            .order_by(EvaluationResultRecord.id)
        )
    ).all()
    snapshot = await session.get(RunFailure, PRIMARY_DEMO_RUN_ID)

print("evaluation_results rows for the policy demo run")
table([dict(row._mapping) for row in history_rows])

print("run_failures snapshot for the same run")
table([
    {
        "run_id": snapshot.run_id,
        "overall_status": snapshot.overall_status,
        "primary_category": snapshot.primary_category,
        "secondary_category": snapshot.secondary_category,
        "max_severity": snapshot.max_severity,
        "classifier_version": snapshot.classifier_version,
        "updated_at": snapshot.updated_at,
    }
])


The policy demo run has been evaluated more than once. The row count grows with each trigger call, but `run_failures` remains one current row for the run.


In [ ]:
async with Session() as session:
    before_history_count = await session.scalar(
        select(func.count())
        .select_from(EvaluationResultRecord)
        .where(EvaluationResultRecord.run_id == PRIMARY_DEMO_RUN_ID)
    )
    before_snapshot_count = await session.scalar(
        select(func.count())
        .select_from(RunFailure)
        .where(RunFailure.run_id == PRIMARY_DEMO_RUN_ID)
    )

async with notebook_client() as client:
    await client.post(f"/v1/runs/{PRIMARY_DEMO_RUN_ID}/evaluate")

async with Session() as session:
    after_history_count = await session.scalar(
        select(func.count())
        .select_from(EvaluationResultRecord)
        .where(EvaluationResultRecord.run_id == PRIMARY_DEMO_RUN_ID)
    )
    after_snapshot_count = await session.scalar(
        select(func.count())
        .select_from(RunFailure)
        .where(RunFailure.run_id == PRIMARY_DEMO_RUN_ID)
    )

table([
    {
        "metric": "evaluation_results rows",
        "before": before_history_count,
        "after": after_history_count,
        "delta": after_history_count - before_history_count,
    },
    {
        "metric": "run_failures rows",
        "before": before_snapshot_count,
        "after": after_snapshot_count,
        "delta": after_snapshot_count - before_snapshot_count,
    },
])


## 7. Run List And Run Detail Read Side

Run list responses expose flat evaluation snapshot fields. Run detail responses expose the nested `failure` block and the latest result per evaluator in `evaluation_summary`.


In [ ]:
async with notebook_client() as client:
    failed_runs = await client.get(
        "/v1/runs",
        params={"overall_status": "fail", "limit": 20},
    )
    policy_runs = await client.get(
        "/v1/runs",
        params={"primary_failure_type": "policy_violation", "limit": 20},
    )
    detail = await client.get(f"/v1/runs/{PRIMARY_DEMO_RUN_ID}")

failed_demo_items = [
    item for item in failed_runs.json()["items"]
    if item["run_id"].startswith(DEMO_PREFIX)
]
policy_demo_items = [
    item for item in policy_runs.json()["items"]
    if item["run_id"].startswith(DEMO_PREFIX)
]
detail_body = detail.json()

print("GET /v1/runs?overall_status=fail")
table([
    {
        "run_id": item["run_id"],
        "status": item["status"],
        "overall_status": item["overall_status"],
        "primary_failure_type": item["primary_failure_type"],
        "max_severity": item["max_severity"],
    }
    for item in failed_demo_items
])

print("GET /v1/runs?primary_failure_type=policy_violation")
table([
    {
        "run_id": item["run_id"],
        "overall_status": item["overall_status"],
        "primary_failure_type": item["primary_failure_type"],
    }
    for item in policy_demo_items
])

print("GET /v1/runs/{run_id} evaluation summary")
print(json.dumps(detail_body["failure"], indent=2))
table(compact_evaluator_result(item) for item in detail_body["evaluation_summary"])


## 8. Failure Analytics

`GET /v1/analytics/failures` excludes never-evaluated runs. It reports evaluated run counts by overall status, then groups failing runs by primary failure type and maximum severity.


In [ ]:
async with notebook_client() as client:
    response = await client.get("/v1/analytics/failures")
    body = response.json()

print(json.dumps(body["run_counts"], indent=2))

print("by_failure_type")
table(body["by_failure_type"])

print("by_severity")
table(body["by_severity"])


The analytics endpoint is global unless you provide `started_after` or `started_before`. For a notebook-specific view, the direct query below shows the same concepts scoped to the demo run IDs.


In [ ]:
async with Session() as session:
    evaluated_demo_count = await session.scalar(
        select(func.count())
        .select_from(RunFailure)
        .where(RunFailure.run_id.in_(DEMO_RUN_IDS))
    )
    total_demo_count = await session.scalar(
        select(func.count())
        .select_from(AgentRun)
        .where(AgentRun.run_id.in_(DEMO_RUN_IDS))
    )
    rows = (
        await session.execute(
            select(
                RunFailure.overall_status,
                RunFailure.primary_category,
                RunFailure.max_severity,
                func.count().label("count"),
            )
            .where(RunFailure.run_id.in_(DEMO_RUN_IDS))
            .group_by(RunFailure.overall_status, RunFailure.primary_category, RunFailure.max_severity)
            .order_by(RunFailure.overall_status, RunFailure.primary_category)
        )
    ).all()

table([
    {
        "demo_agent_runs": total_demo_count,
        "demo_evaluated_runs": evaluated_demo_count,
        "demo_never_evaluated_runs": total_demo_count - evaluated_demo_count,
    }
])

table([dict(row._mapping) for row in rows])


## 9. Overview Analytics

Phase 5 extends overview analytics with behavioral pass rate while preserving runtime success rate as a separate metric. Runtime success comes from `agent_runs.status`; behavioral pass comes from the latest `run_failures.overall_status` rows.


In [ ]:
async with notebook_client() as client:
    response = await client.get("/v1/analytics/overview")
    body = response.json()

print(json.dumps({
    "runtime_success_rate": body["runtime_success_rate"],
    "behavioral_pass_rate": body["behavioral_pass_rate"],
    "run_counts": body["run_counts"],
    "evaluation_counts": body["evaluation_counts"],
}, indent=2))


## 10. Cleanup

This removes only the notebook demo run IDs and disposes the database engine.


In [ ]:
async with Session() as session:
    await delete_demo_runs(session)

await engine.dispose()
display(Markdown("Notebook demo rows removed and engine disposed."))
